In [1]:
import os
import mesa
import time
import asyncio
import nest_asyncio
from dotenv import load_dotenv
from google import genai

nest_asyncio.apply()

load_dotenv(override=True)
api_key = os.environ.get("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)

class AsyncAgent(mesa.Agent):
    def __init__(self, model):
        super().__init__(model)
        self.response_time = 0
        
    async def async_think(self):
        prompt = f"You are Agent {self.unique_id}. Respond strictly with: 'Agent {self.unique_id} online.'"
        
        start = time.time()
        try:
            response = await client.aio.models.generate_content(
                model='gemini-2.5-flash',
                contents=prompt,
            )
            end = time.time()
            self.response_time = end - start
            print(f"✅ Agent {self.unique_id} finished in {self.response_time:.2f}s: {response.text.strip()}")
            
        except Exception as e:
            print(f"❌ Agent {self.unique_id} failed: {e}")

class AsyncModel(mesa.Model):
    def __init__(self, n_agents):
        super().__init__()
        for i in range(n_agents):
            agent = AsyncAgent(self)
            
    async def run_all_agents(self):
        tasks = [agent.async_think() for agent in self.agents]
        await asyncio.gather(*tasks)

    def step(self):
        print(f"\n--- Initiating Concurrent Step {self.steps + 1} ---")
        start_step = time.time()

        asyncio.run(self.run_all_agents()) 
        
        end_step = time.time()
        total_time = end_step - start_step
        
        print("-" * 40)
        print(f"⏱️ TOTAL STEP TIME (3 Agents): {total_time:.2f} seconds")
        print("-" * 40)
        self.steps += 1

In [2]:
model = AsyncModel(3)
model.step()


--- Initiating Concurrent Step 2 ---
❌ Agent 2 failed: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 38.131979277s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'locatio